# 05 · Model Karşılaştırma Eğitimleri

**Amaç:** `configs/experiments.json` içinde tanımlı mimarileri aynı veri bölmesi, aynı sınıf ağırlığı ve aynı erken durdurma kuralıyla eğitip Drive'a kaydetmek.

* Bu defter **sadece eğitim** yapar; karşılaştırma tablosu ve grafikler `06_compare_results.ipynb` defterinde üretilir.
* Her model **bağımsız bir hücredir**; istediğinizi tek tek çalıştırabilirsiniz.
* `results.json` zaten varsa hücre o modeli atlar; eğitim yarıda kalmışsa `last.pt` üzerinden kaldığı turdan devam eder.
* **Test kümesi bu defterde hiç kullanılmaz.** Eşik doğrulama kümesinde seçildiği için buradaki eşikli metrikler (F1, Precision, Recall) hafifçe iyimser olabilir; test aşamasında bu eşik sabit tutulacaktır.

| Hücre | Deney klasörü | Model | Durum | Amaç |
|---|---|---|---|---|
| 1 | `05_cnn_avg` | `TornadoCNN (pool="avg")` | Zorunlu | Global Max vs Global Avg havuzlama ablasyonu |
| 2 | `05_resnet18_pretrained` | `ResNet-18 (ön-eğitimli, lr=1e-4)` | Zorunlu | Transfer öğrenme karşılaştırması |
| 3 | `05_resnet18_scratch` | `ResNet-18 (sıfırdan, lr=1e-4)` | Zorunlu | Ön-eğitimin etkisini aynı `lr` ile ölçme |
| 4 | `05_cnn_bn` | `TornadoCNN (pool="max", batch_norm=True)` | İsteğe bağlı | BatchNorm etkisi |
| 5 | Şablon | Toplu veya çoklu seed koşumu | İsteğe bağlı | Config'deki yeni/ek deneyler |

In [1]:
# --- Kurulum: hem Google Colab'da hem kendi bilgisayarında çalışır ---
import json, os, sys
from pathlib import Path

BOOTSTRAP_BRANCH = os.environ.get("TORNADO_BRANCH", "m")

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, userdata
    drive.mount("/content/drive")                     # işlenmiş veri ve modeller Drive'da saklanır
    REPO = Path("/content/tornado-cnn")
    cfg_file = REPO / "configs" / "experiments.json"
    branch = json.loads(cfg_file.read_text())["repo"]["branch"] if cfg_file.exists() else BOOTSTRAP_BRANCH
    if not REPO.exists():
        try:                                          # repo private ise: Colab Secrets'taki token
            token = userdata.get("GITHUB_TOKEN") + "@"
        except Exception:                             # repo public ise token gerekmez
            token = ""
        !git clone -q -b {branch} https://{token}github.com/baevv/tornado-cnn.git {REPO}
    else:
        !git -C {REPO} fetch -q origin {branch}
        !git -C {REPO} checkout -q {branch}
        !git -C {REPO} pull -q origin {branch}
    os.chdir(REPO)
else:
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    cfg_file = REPO / "configs" / "experiments.json"
    branch = json.loads(cfg_file.read_text())["repo"]["branch"] if cfg_file.exists() else BOOTSTRAP_BRANCH

sys.path.insert(0, str(REPO))
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("Colab" if IN_COLAB else "Yerel", "| repo:", REPO, "| dal:", branch)

Mounted at /content/drive
Colab | repo: /content/tornado-cnn | dal: m


In [2]:
import torch
from src.helper import model as cnn
from src.helper import paths, train

cfg_data = paths.load_config("data.json")
cfg = paths.load_config("train.json")
exp_registry = paths.load_config("experiments.json")
EXPERIMENTS = {e["name"]: e for e in exp_registry["experiments"]}

device = train.get_device()
if device.type == "cuda":
    print("cihaz: GPU |", torch.cuda.get_device_name(0))
else:
    print("UYARI: GPU yok, eğitim çok yavaş olur. Çalışma zamanı → T4 GPU seçin.")
print("Tanımlı deneyler:", list(EXPERIMENTS))
cfg

cihaz: GPU | Tesla T4
Tanımlı deneyler: ['05_cnn_avg', '05_resnet18_pretrained', '05_resnet18_scratch', '05_cnn_bn']


{'batch_size': 64,
 'optimizer': 'adam',
 'lr': 0.001,
 'weight_decay': 0.0,
 'epochs': 20,
 'patience': 4,
 'dropout': 0.3,
 'weighting': 'pos_weight',
 'seed': 42,
 'num_workers': 2}

In [3]:
HIZLI_DENEME = True      # True: sadece 2017 ile hızlı deneme

data_root, models_root = paths.prepare_colab_dirs()
print("veri klasörü:", data_root, "| model kayıt klasörü:", models_root)

years = [2017] if HIZLI_DENEME else cfg_data["years"]
splits = train.load_splits(data_root)
local_dir = Path("/content/data_local") if IN_COLAB else None
data, variables = train.load_data(paths.processed_dir(), splits, years, which=("train", "val"), local_dir=local_dir)
y_val = data["val"]["y"]

print(f"yıllar: {years} | kanallar: {variables} | girdi: {data['train']['X'].shape[1:]}")
print(f"eğitim: {len(data['train']['y']):,} | doğrulama: {len(y_val):,}")

veri klasörü: /content/drive/MyDrive/tornado-cnn/data | model kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models
yıllar: [2017] | kanallar: ['DBZ', 'VEL', 'WIDTH'] | girdi: (3, 120, 240)
eğitim: 14,311 | doğrulama: 2,602


## Ortak eğitim fonksiyonu

Her model hücresi ayarını `configs/experiments.json` dosyasından okuyan `run_from_config(name)` fonksiyonunu çağırır:
1. Hedef klasörde `results.json` zaten varsa modeli yeniden eğitmeden atlar (`force=True` verilmedikçe).
2. Seçilen `seed` değerini yeniden kurar (aynı batch karıştırması = adil karşılaştırma).
3. `cnn.build_model(arch, ...)` ile modeli kurar ve `train.fit` ile eğitir.
4. Doğrulama kümesinde F1'i en yüksek yapan eşiği seçip `results.json` kaydeder. *(Not: Eşik aynı `val` kümesinde seçildiği için buradaki F1/Precision/Recall değerleri hafifçe iyimserdir; asıl tarafsız ölçüm test aşamasında bu eşik sabitlenerek yapılır.)*

In [4]:
def run_experiment(name: str, arch: str, model_kwargs=None, train_overrides=None,
                   arch_label=None, note="", force=False):
    out_name = f"{name}_hizli" if HIZLI_DENEME else name
    out_dir = paths.models_dir(out_name)
    if not force and (out_dir / "results.json").exists():
        print(f"ATLANDI: {out_dir / 'results.json'} zaten var (yeniden eğitmek için force=True verin).")
        return

    exp_cfg = {**cfg, **(train_overrides or {})}
    unknown = set(exp_cfg) - set(cfg)
    if unknown:
        raise ValueError(f"train.json içinde olmayan bilinmeyen ayarlar: {unknown}")

    train.set_seed(exp_cfg["seed"])
    w_train, pos_weight = train.class_weights(data["train"]["meta"], exp_cfg["weighting"])
    train_loader, val_loader = train.make_loaders(data, exp_cfg, w_train)

    net = cnn.build_model(
        arch, in_channels=len(variables), dropout=exp_cfg["dropout"], **(model_kwargs or {})
    ).to(device)
    n_params = cnn.count_params(net)

    print(f"=== {arch_label or name} | parametre: {n_params:,} | lr: {exp_cfg['lr']} | "
          f"weighting: {exp_cfg['weighting']} | seed: {exp_cfg['seed']} ===")
    print("kayıt klasörü:", out_dir)

    history = train.fit(net, train_loader, val_loader, exp_cfg, variables, out_dir, pos_weight, device)
    p_val, val_loss = train.predict(net, val_loader, variables, device, pos_weight)
    threshold = train.choose_threshold(y_val, p_val)
    val_metrics = train.metrics(y_val, p_val, threshold)
    best_epoch = int(history.loc[history["doğrulama AUC"].idxmax(), "tur"])

    train.save_results(
        out_dir,
        model_name=arch_label or name,
        note=note,
        params=n_params,
        config=exp_cfg,
        years=years,
        variables=variables,
        pos_weight=pos_weight,
        best_epoch=best_epoch,
        threshold=threshold,
        val_loss=float(val_loss),
        validation=val_metrics,
    )
    print(f"Tamamlandı → en iyi tur: {best_epoch} | PR-AUC: {val_metrics['PR-AUC']:.4f} | "
          f"ROC-AUC: {val_metrics['ROC-AUC']:.4f} | F1 (val eşikli): {val_metrics['F1']:.4f}\n")

    del net
    if device.type == "cuda":
        torch.cuda.empty_cache()


def run_from_config(name: str, seed=None, force=False):
    spec = EXPERIMENTS[name]
    overrides = dict(spec.get("train", {}))
    if seed is not None:
        overrides["seed"] = seed
    return run_experiment(
        name=spec["name"],
        arch=spec["arch"],
        model_kwargs=spec.get("model_kwargs", {}),
        train_overrides=overrides,
        arch_label=spec.get("label", spec["name"]),
        note=spec.get("note", ""),
        force=force,
    )

## 1. TornadoCNN — Global Average Pooling (`05_cnn_avg`)

Zorunlu ablasyon: `03_baseline` ile birebir aynı ayarlar, yalnızca `AdaptiveMaxPool2d(1)` yerine `AdaptiveAvgPool2d(1)`.

In [5]:
run_from_config("05_cnn_avg")

=== TornadoCNN (Global Avg) | parametre: 105,761 | lr: 0.001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_cnn_avg_hizli
tur  1 | eğitim kaybı 1.1444 | doğrulama kaybı 1.0370 | AUC 0.7539 | PR-AUC 0.1781 | 12 sn  ★ en iyi
tur  2 | eğitim kaybı 1.0798 | doğrulama kaybı 1.0179 | AUC 0.7734 | PR-AUC 0.1978 | 8 sn  ★ en iyi
tur  3 | eğitim kaybı 1.0662 | doğrulama kaybı 1.0472 | AUC 0.7814 | PR-AUC 0.2000 | 8 sn  ★ en iyi
tur  4 | eğitim kaybı 1.0463 | doğrulama kaybı 1.0090 | AUC 0.7703 | PR-AUC 0.2055 | 8 sn
tur  5 | eğitim kaybı 1.0314 | doğrulama kaybı 0.9912 | AUC 0.7872 | PR-AUC 0.2345 | 8 sn  ★ en iyi
tur  6 | eğitim kaybı 1.0238 | doğrulama kaybı 0.9879 | AUC 0.7900 | PR-AUC 0.2548 | 8 sn  ★ en iyi
tur  7 | eğitim kaybı 1.0064 | doğrulama kaybı 0.9543 | AUC 0.8111 | PR-AUC 0.2427 | 8 sn  ★ en iyi
tur  8 | eğitim kaybı 0.9537 | doğrulama kaybı 0.9018 | AUC 0.8321 | PR-AUC 0.3022 | 8 sn  ★ en iyi
tur  9 | eğitim kaybı 0.9133 | doğr

## 2. ResNet-18 — Ön-eğitimli (`05_resnet18_pretrained`)

Zorunlu ikinci mimari: ImageNet ön-eğitimli ResNet-18 (`lr=1e-4`, `[-1, 1]` radar ölçeği).

In [6]:
run_from_config("05_resnet18_pretrained")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 191MB/s]


=== ResNet-18 (ön-eğitimli) | parametre: 11,177,025 | lr: 0.0001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_resnet18_pretrained_hizli
tur  1 | eğitim kaybı 0.9038 | doğrulama kaybı 1.1029 | AUC 0.7637 | PR-AUC 0.2438 | 29 sn  ★ en iyi
tur  2 | eğitim kaybı 0.3889 | doğrulama kaybı 1.5641 | AUC 0.7639 | PR-AUC 0.3179 | 28 sn  ★ en iyi
tur  3 | eğitim kaybı 0.1618 | doğrulama kaybı 1.3154 | AUC 0.8002 | PR-AUC 0.3520 | 28 sn  ★ en iyi
tur  4 | eğitim kaybı 0.0963 | doğrulama kaybı 2.0305 | AUC 0.7897 | PR-AUC 0.4069 | 28 sn
tur  5 | eğitim kaybı 0.0651 | doğrulama kaybı 1.6624 | AUC 0.8159 | PR-AUC 0.4101 | 28 sn  ★ en iyi
tur  6 | eğitim kaybı 0.0493 | doğrulama kaybı 2.0171 | AUC 0.8189 | PR-AUC 0.4103 | 28 sn  ★ en iyi
tur  7 | eğitim kaybı 0.0487 | doğrulama kaybı 2.1477 | AUC 0.8164 | PR-AUC 0.3751 | 28 sn
tur  8 | eğitim kaybı 0.0207 | doğrulama kaybı 2.0521 | AUC 0.8010 | PR-AUC 0.3962 | 28 sn
tur  9 | eğitim kaybı 0.0076 | d

## 3. ResNet-18 — Sıfırdan (`05_resnet18_scratch`)

Zorunlu karşılaştırma: Ön-eğitimin katkısını adil ölçmek için `05_resnet18_pretrained` ile aynı `lr=1e-4`.

In [7]:
run_from_config("05_resnet18_scratch")

=== ResNet-18 (sıfırdan) | parametre: 11,177,025 | lr: 0.0001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_resnet18_scratch_hizli
tur  1 | eğitim kaybı 1.0401 | doğrulama kaybı 1.0557 | AUC 0.7578 | PR-AUC 0.2750 | 28 sn  ★ en iyi
tur  2 | eğitim kaybı 0.8006 | doğrulama kaybı 1.0557 | AUC 0.7542 | PR-AUC 0.2671 | 28 sn
tur  3 | eğitim kaybı 0.5636 | doğrulama kaybı 1.2608 | AUC 0.7349 | PR-AUC 0.2989 | 28 sn
tur  4 | eğitim kaybı 0.3850 | doğrulama kaybı 1.3300 | AUC 0.7631 | PR-AUC 0.3139 | 28 sn  ★ en iyi
tur  5 | eğitim kaybı 0.2085 | doğrulama kaybı 1.8574 | AUC 0.7593 | PR-AUC 0.2997 | 28 sn
tur  6 | eğitim kaybı 0.1331 | doğrulama kaybı 2.0938 | AUC 0.7520 | PR-AUC 0.2455 | 28 sn
tur  7 | eğitim kaybı 0.1104 | doğrulama kaybı 1.8454 | AUC 0.7657 | PR-AUC 0.2668 | 28 sn  ★ en iyi
tur  8 | eğitim kaybı 0.1438 | doğrulama kaybı 2.4834 | AUC 0.7410 | PR-AUC 0.3093 | 28 sn
tur  9 | eğitim kaybı 0.0985 | doğrulama kaybı 2.8978 | AU

## 4. TornadoCNN + BatchNorm (`05_cnn_bn`, isteğe bağlı)

Evrişim bloklarına `BatchNorm2d` eklemenin yakınsama hızı ve doğrulama kaybı üzerindeki etkisini ölçer.

In [8]:
run_from_config("05_cnn_bn")

=== TornadoCNN (Global Max + BN) | parametre: 106,241 | lr: 0.001 | weighting: pos_weight | seed: 42 ===
kayıt klasörü: /content/drive/MyDrive/tornado-cnn/models/05_cnn_bn_hizli
tur  1 | eğitim kaybı 1.0564 | doğrulama kaybı 0.9646 | AUC 0.7829 | PR-AUC 0.2714 | 11 sn  ★ en iyi
tur  2 | eğitim kaybı 0.8972 | doğrulama kaybı 0.9191 | AUC 0.8104 | PR-AUC 0.3810 | 11 sn  ★ en iyi
tur  3 | eğitim kaybı 0.8602 | doğrulama kaybı 0.9029 | AUC 0.8443 | PR-AUC 0.4090 | 11 sn  ★ en iyi
tur  4 | eğitim kaybı 0.8198 | doğrulama kaybı 0.9931 | AUC 0.8202 | PR-AUC 0.4046 | 11 sn
tur  5 | eğitim kaybı 0.7353 | doğrulama kaybı 0.9548 | AUC 0.8209 | PR-AUC 0.4188 | 11 sn
tur  6 | eğitim kaybı 0.6992 | doğrulama kaybı 0.8369 | AUC 0.8489 | PR-AUC 0.4724 | 11 sn  ★ en iyi
tur  7 | eğitim kaybı 0.6333 | doğrulama kaybı 0.8451 | AUC 0.8710 | PR-AUC 0.5149 | 11 sn  ★ en iyi
tur  8 | eğitim kaybı 0.5712 | doğrulama kaybı 1.1423 | AUC 0.8175 | PR-AUC 0.4366 | 11 sn
tur  9 | eğitim kaybı 0.5294 | doğrulama kay

## 5. Toplu veya çoklu seed koşumu (isteğe bağlı)

`configs/experiments.json` dosyasına eklediğin yeni deneyleri tek satırda çalıştırmak ya da finalistleri ek seed değerleriyle tekrarlamak için kullanabilirsin (bitmiş deneyler otomatik atlanır).

In [9]:
# Config'deki tüm zorunlu deneyleri tek seferde çalıştırmak için:
# for exp in exp_registry["experiments"]:
#     if exp.get("required", True):
#         run_from_config(exp["name"])

# Finalistler için çoklu seed örneği (aynı klasörü ezmemek için yeni adla experiments.json'a eklemek en temizidir):
# for s in (123, 7):
#     run_experiment(
#         name=f"05_cnn_max_s{s}",
#         arch="tornado_cnn",
#         model_kwargs={"pool": "max"},
#         train_overrides={"seed": s},
#         arch_label=f"TornadoCNN (Global Max, seed={s})",
#     )